# Setup

In [18]:
!pip install jax flax optax tokenizers datasets

In [19]:
# SETUP

# datasets für das Laden der Daten (Wikipedia crawled)
from datasets import load_dataset

# tokenizers für das Encoden von den Texten
from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.trainers import WordLevelTrainer

class Word2VecDataPipeline:

    def __init__(self, window_size: int, vocab_size: int = 32768):
        self.window_size = window_size
        self.target_vocab_size = vocab_size
        self.tokenizer = None
        self.vocab_size = None

    def load_and_prepare_all(self):
        dataset = load_dataset("wikitext", "wikitext-103-v1", split="train", streaming=True)

        # Tokenizer: Transform words into numbers (i.e., Ordinal Encoding)
        self.tokenizer = Tokenizer(WordLevel(unk_token="<unk>"))
        self.tokenizer.pre_tokenizer = Whitespace()
        trainer = WordLevelTrainer(
            special_tokens=["<unk>"],
            min_frequency=0,
            vocab_size=self.target_vocab_size
        )

        # Take first 1m lines to build vocab (faster & saves RAM)
        def batch_iterator():
            for i, item in enumerate(dataset):
                yield item["text"]
                if i > 1_000_000: break

        self.tokenizer.train_from_iterator(batch_iterator(), trainer=trainer)
        self.vocab_size = self.tokenizer.get_vocab_size()

    def get_batches(self, split: str, batch_size: int, skip_items = 0):
        """
        :param split: 'train' or 'valid'
        :param batch_size:
        """
        dataset = load_dataset("wikitext", "wikitext-103-v1", split=split, streaming=True)

        xs, ys = [], []
        for item in dataset:
            # Skip empty texts
            text = item["text"].strip()
            if not text: continue

            # Skip texts that have too few words
            ids = self.tokenizer.encode(text).ids
            if len(ids) < self.window_size * 2 + 1: continue

            # Allow user to skip $skip_items$ items
            if skip_items > 0: skip_items -= 1; continue

            # Loop over and aggregate all valid windows in current item
            for i in range(len(ids) - self.window_size * 2):
                # left window + middle token + right window
                full_window_size = self.window_size + 1 + self.window_size
                window_ids = ids[i:i+full_window_size]
                y = window_ids[self.window_size]
                x = window_ids[:self.window_size] + window_ids[self.window_size + 1:]
                ys.append(y)
                xs.append(x)

                # Yield/Return accumulated batch
                if len(ys) == batch_size:
                    yield jnp.array(xs), jnp.array(ys)
                    xs, ys = [], [] # reset for next batch

    def encode(self, text: str) -> list[int]:
        """
        :param text: Text to encode
        :return: List of tokens (here: words ordinal encoded)
        """
        return self.tokenizer.encode(text).ids

    def decode(self, ids: list[int]) -> str:
        """ inverse of encode """
        return self.tokenizer.decode(ids, skip_special_tokens=False)

    def demonstrate_cbow_window(self, text: str):
        """  """
        ids = self.encode(text)
        if len(ids) < self.window_size * 2 + 1:
            return []

        results = []
        for i in range(len(ids) - self.window_size * 2):

            # Extractg CBOW window
            window_ids = ids[i : i + self.window_size * 2 + 1]
            y = window_ids[self.window_size]
            x = window_ids[:self.window_size] + window_ids[self.window_size + 1:]

            # token (ids) back to words
            x_words = [self.decode([token_id]) for token_id in x]
            y_word = self.decode([y])

            results.append((x_words, y_word))
        return results

def show_debug_examples(pipeline, params):
    def load_demos(pipeline, num_demos: int, skip_items: int):
        demo_contexts, demo_targets = [], []
        val_stream = pipeline.get_batches("validation", batch_size=1, skip_items=skip_items)
        for context_batch, target_batch in val_stream:
            demo_contexts.append(context_batch[0])
            demo_targets.append(target_batch[0])
            if len(demo_contexts) >= num_demos:
                break
        return jnp.array(demo_contexts), jnp.array(demo_targets)
    for item_id in range(10):
      x_demos, y_demos = load_demos(pipeline, num_demos=3, skip_items=item_id)
      y_hat_demos = predict_step(params, x_demos)
      for x, y, y_hat in zip(x_demos, y_demos, y_hat_demos):
          x_words = [pipeline.decode([token_id]) for token_id in x]
          assert len(x_words) == WINDOW_SIZE * 2
          y_word = pipeline.decode([y])
          y_hat_word = pipeline.decode([y_hat])
          print(f"Context: {x_words[:WINDOW_SIZE]} || {x_words[WINDOW_SIZE:]}  -->  GT: '{y_word}' | Pred: '{y_hat_word}'")
      print("-" * 30 + "\n")
    print("-" * 60 + "\n")

# (Extra) Word2Vec – Aufgabenblatt 6

**Dieses Aufgabenblatt ist anders als die bisherigen Aufgabenblätter. Ihr müsst keine Zeile Code selber schreiben, sondern meinen Code lesen und verstehen. Versucht dabei möglichst viel vom Code zu verstehen und mit dem Theorieteil zu verknüpften.**

In dieser Aufgabe trainieren wir unser eigenes Word2Vec-Modell mit **FLAX**.

Um den Fokus auf die Kernkonzepte zu legen, ist ein Grossteil des Codes bereits implementiert. Das Hauptziel dieses Aufgabenblatts besteht darin, die Implementierung im Detail zu verstehen und die `Verbindung zum Theorieteil` herzustellen.

Da das Training eines vollständigen Word2Vec-Modells bis zur Konvergenz mehrere Stunden in Anspruch nehmen kann, werden wir hier nur einen Ausschnitt bearbeiten. Dennoch bietet die Aufgabe ein wertvolles praktisches Gespür für die Arbeitsweise von Deep Learning in der Realität.

### Warum Word2Vec?
Obwohl Word2Vec bereits im Jahr 2013 veröffentlicht wurde, illustriert es auch heute noch hervorragend, wie hochwertige `Repräsentationen (Embeddings)` aus grossen Datenmengen gelernt werden können. Die Fähigkeit, semantische Strukturen aus unstrukturierten Daten (wie Texten oder Bildern) zu extrahieren, ist das Fundament für den Erfolg des modernen Deep Learning. Word2Vec dient uns somit als klassisches und zugleich sehr anschauliches Beispiel für dieses Prinzip.

## Aufgabe 1 - Setup in Google Colab

Für das Training des Modells in Aufgabe 4 benötigen wir eine **GPU**. Google bietet über Colab kostenlos Zugriff auf GPU-Rechenleistung an.

1. Öffne dieses Notebook in **Google Colab** mittels Klick auf "Open in CoLab": <a href="https://colab.research.google.com/github/benikm91/cas_machine-learning-exercise/blob/main/src/word2vec/word2vec.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
2. In CoLab: Ändere den Laufzeittyp auf **`T4 GPU`** unter `Laufzeit -> Laufzeittyp ändern`, wie in [diesem Video](https://www.youtube.com/watch?v=6H381fUOolU) gezeigt.

*Hinweis: Falls Colab bei Ihnen nicht funktioniert, können Sie die Aufgaben 2 und 3 lokal (ohne GPU) bearbeiten.*

## Aufgabe 2 - Datensatz

In dieser Aufgabe bereiten wir den Datensatz vor. Dazu verwenden wir die Python libraries `datasets` und `tokenizers` (Siehe `Word2VecDataPipeline` in Setup für Details).

1. In der unteren Code-Zelle `Load Data` laden wir die Daten, die wir dann für das Trainieren vom Modell verwenden. Führen Sie diese Zelle aus, dies kann einige Minute dauern.
2. Führe auch die nächste Code-Zelle `Concept Demonstration` aus. Lesen Sie die Kommentare und versuchen Sie zu verstehen, was passiert.

In [20]:
# Load Data

# Hier definieren wir die Fenstergrösse für CBOW.
# Im Training muss Word2Vec das Wort anhand des linken und rechten Fensters vorhersagen.
# Hier also die 4 Wörter vor dem Wort und die 4 Wörter nach dem Wort.
WINDOW_SIZE = 4

print("Dataset wird heruntergeladen... Kann beim ersten Ausfähren ein paar Minuten dauern...")
pipeline = Word2VecDataPipeline(window_size=WINDOW_SIZE)
pipeline.load_and_prepare_all()

print(f"Vocabular Grösse ist (künstlich) beschränkt auf die häufigsten {pipeline.vocab_size} Wörter.")

Dataset wird heruntergeladen... Kann beim ersten Ausfähren ein paar Minuten dauern...
Vocabular Grösse ist (künstlich) beschränkt auf die häufigsten 32768 Wörter.


In [21]:
# Concept Demonstration

sample_sentence = "the quick brown fox jumps over the lazy dog" # Beispielsatz
print(f"Beispielsatz: {sample_sentence}")
# Mit `pipeline.encode` transformieren wir den Text zu Zahlen.
# `encode` teilt dabei den Text nach Leerzeichen zu Wörtern auf und Ordinal Encoded die Wörter zu Zahlen (nach Vocabulary).
print(f"Encoded IDs:   {pipeline.encode(sample_sentence)}")

# Aus einem Satz erstellt die `pipeline` dann alle möglichen Paare von Context (linke und rechte Wörter) und Target (Wort).
print("\nCBOW Sliding Window Beispiele")
for ctx, tgt in pipeline.demonstrate_cbow_window(sample_sentence):
    # Aus dem Context muss das Target hervorgesagt werden.
    print(f"Context: {ctx}  -->  Target: '{tgt}'")
    # Das Target ist das Wort inmitten des Context's
    print(f"Linker Context: {ctx[:WINDOW_SIZE]} || Target: '{tgt}' || Rechter Context: {ctx[WINDOW_SIZE:]}")

raw_word = 'petrichor' #  Beispiel seltenes Wort
raw_word_token = pipeline.encode(raw_word)
print(f"Beispiel seltenes Wort:  {raw_word}")
print(f"Seltenes Wort als Token encoded:   {raw_word_token}")
# Beachte, decoded ist es "<unk>"" und nicht "petrichor", da "petrichor" zu selten und deshalb nicht Teil unseres Vocabular wurde.
# Alle seltenen Wörter werden auf "<unk>" gemappt.
print(f"Seltenes Wort Token wieder decoded:   {pipeline.decode(raw_word_token)}")


Beispielsatz: the quick brown fox jumps over the lazy dog
Encoded IDs:   [1, 3845, 2053, 11184, 11896, 74, 1, 18608, 3678]

CBOW Sliding Window Beispiele
Context: ['the', 'quick', 'brown', 'fox', 'over', 'the', 'lazy', 'dog']  -->  Target: 'jumps'
Linker Context: ['the', 'quick', 'brown', 'fox'] || Target: 'jumps' || Rechter Context: ['over', 'the', 'lazy', 'dog']
Beispiel seltenes Wort:  petrichor
Seltenes Wort als Token encoded:   [0]
Seltenes Wort Token wieder decoded:   <unk>


## Aufgabe 3 - Word2Vec Modell

In der Code-Zelle `Word2VecCBOW Modell` sehen Sie ein fast fertig implementiertes Word2Vec Modell.

1. Schauen Sie die Methode `setup` in Ruhe an. Es initialisiert zwei Teile für das Modell. `word_embedding` ist der erste Teil, der die Tokens auf die Repräsentation transformiert, die wir lernen möchten. `output_projection` transformiert die Repräsentation zurück zu den Tokens, spezifisch, zu einer Wahrscheinlichkeitsverteilung über alle Tokens.
2. Schauen Sie die Methode `__call__` in Ruhe an. Der Code macht drei Dinge: (1.) Input Tokens `x` zu Repräsentationen encoden (2.) Die Summe über das Fenster von Inputs nehmen. (3.) Das resultierende Embedding zu Logits über alle Tokens (Vocab) transformieren.
3. Wo im folgenden Bild sind die jeweiligen Schritte von `__call__`wieder zu finden?

![Word2Vec](https://github.com/benikm91/cas_machine-learning-exercise/raw/refs/heads/main/src/word2vec/img/word2vec.png)

In [22]:
# Word2VecCBOW Modell

import flax.linen as nn

class Word2VecCBOW(nn.Module):
    vocab_size: int
    embed_dimension: int

    def setup(self):
        self.word_embeddings = nn.Embed(
            num_embeddings=self.vocab_size,
            features=self.embed_dimension
        )
        self.output_projection = nn.Dense(features=self.vocab_size)

    def __call__(self, x):
        # x shape: (batch_size, window_size * 2)
        embeds = self.word_embeddings(x)    # 1. (batch_size, window_size * 2, embed_dimension)
        z = embeds.sum(axis=1)              # 2. (batch_size, embed_dimension)
        logits = self.output_projection(z)  # 3. (batch_size, vocab_size)
        return logits

## Aufgabe 4

1. Lass folgende Code-Zelle laufen.
2. Alle 10'000 Schritten (nach 5'120'000 CBOW-Beispielen!), lassen wir das Modell auf Beispiele aus den Validiernugsdaten laufen. Der Output sieht dabei so aus: `Context: ['Homarus', '<unk>', ',', 'known'] || ['the', 'European', 'lobster', 'or']  -->  GT: 'as' | Pred: 'as'` und ist folgendermassen zu verstehen. Das Modell sieht folgenden Kontext `... Homarus <unk>, known __ the European lobster or ...` und muss das Wort an `__` vorhersagen. `GT: 'as'` bedeutet das Wort war auf Wikipedia "as" und `Pred: 'as'` bedeutet das Modell gibt dem Wort "as" die grösste Warhscheinlichkeit. Versuche den Output über das Training zu verfolgen. Wir sollten sehen, dass der `Train Loss` runter geht. Das Modell braucht aber lange bis es gut wird. d

In [ ]:
import jax
import jax.numpy as jnp
import optax
from tqdm import tqdm

def cross_entropy_loss(y_hat, y):
    y_one_hot = jax.nn.one_hot(y, num_classes=y_hat.shape[-1])
    return optax.softmax_cross_entropy(logits=y_hat, labels=y_one_hot).mean()

@jax.jit
def train_step(params, x, y):
    def loss_fn(params):
        bound_model = model.bind({'params': params})
        y_hat = bound_model(x)
        loss = cross_entropy_loss(y_hat, y)
        return loss

    # Calculate gradients (loss_fn is automatically derived by JAX using automatic differentiation)
    loss_value, gradients = jax.value_and_grad(loss_fn)(params)

     # Gradient Descent update
    new_params = jax.tree_util.tree_map(
        lambda p, g: p - LEARNING_RATE * g,
        params,
        gradients
    )

    return new_params, loss_value

@jax.jit
def predict_step(params, x):
    bound_model = model.bind({'params': params})
    y_hat = bound_model(x)
    return jnp.argmax(y_hat, axis=-1)

@jax.jit
def eval_step(params, x, y):
    bound_model = model.bind({'params': params})
    y_hat = bound_model(x)
    return cross_entropy_loss(y_hat, y)

# Check ob eine GPU vorhanden ist
devices = jax.devices()
gpu_found = any(device.platform == 'gpu' for device in devices)

if not gpu_found:
    print("❌ FEHLER: Keine GPU gefunden!")
    print("Bitte prüfen Sie Aufgabe 1: Haben Sie den Laufzeittyp auf 'T4 GPU' umgestellt?")
    print(f"Aktuelle Geräte: {devices}")
    print("Training ist auf der CPU sehr langsam...")

# --- Cell 3: Training Initialization ---
BATCH_SIZE = 512
MAX_EPOCH = 10
LEARNING_RATE = 0.001
LATENT_DIM = 300

model = Word2VecCBOW(vocab_size=pipeline.vocab_size, embed_dimension=LATENT_DIM)
initial_params = model.init(jax.random.PRNGKey(42), jnp.ones((1, pipeline.window_size * 2), jnp.int32))['params']

print("\nStarting JAX training (compiled via XLA)...")
params = initial_params
for epoch in range(1, MAX_EPOCH + 1):
    train_loss = 0.0
    num_batches = 0

    # Train Loop
    train_batches = pipeline.get_batches("train", BATCH_SIZE)
    for batch, y in tqdm(train_batches, total=183_320, desc=f"Epoch {epoch} Train"):
        params, loss_val = train_step(params, batch, y)
        train_loss += loss_val
        num_batches += 1

        if num_batches % 10_000 == 0:
            show_debug_examples(pipeline, params)
            print(f"Epoch {epoch} - Step {num_batches} - Train loss: {train_loss / num_batches}")

    # Evaluation on validation data
    val_loss = 0.0
    val_num_batches = 0
    val_batches = pipeline.get_batches("validation", BATCH_SIZE, shuffle=False)
    for batch, y in tqdm(val_batches, desc=f"Epoch {epoch} Eval "):
        loss_val = eval_step(params, batch, y)
        val_loss += loss_val
        val_num_batches += 1

    print(f"=== Epoch {epoch} Summary ===")
    print(f"Train Loss: {train_loss / num_batches:.4f}")
    print(f"Val Loss:   {val_loss / val_num_batches:.4f}\n")


❌ FEHLER: Keine GPU gefunden!
Bitte prüfen Sie Aufgabe 1: Haben Sie den Laufzeittyp auf 'T4 GPU' umgestellt?
Aktuelle Geräte: [CpuDevice(id=0)]
Training ist auf der CPU sehr langsam...

Starting JAX training (compiled via XLA)...


Epoch 1 Train:   0%|          | 31/183320 [00:29<64:44:02,  1.27s/it]

## Schlusswort - Aufgabenblatt 6

Wir haben hier Einblicke in das Trainieren von Word2Vec gesehen. Ziel ist es einen Einblick in Deep Learning in der Praxis zu bekommen.

Als nächstes könnte man das Modell fertig trainieren (dauert ein paar Stunden), speichern, und dann die gelernten Repräsentationen untersuchen. Ich habe dies vorlängerer Zeit (und in Scala) gemacht, siehe [Repo](https://github.com/benikm91/unibasel_word2vec_typesafe).